In [ ]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
#as_frame=False returns nd.array, otherwise it will return pd.dataframe (default)
mnist.keys()
#there are 784 pixels (28*28 images)

In [ ]:
mnist["data"][:5]

In [ ]:
mnist["target"][:5] #targets are the labels

In [ ]:
X, y = mnist["data"], mnist["target"]
X.shape

In [ ]:
y.shape

In [ ]:
type(X)
#when indexing pd.dataframe, use .loc[n]

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

some_digit = X[0]
some_digit_img = some_digit.reshape(28,28)

plt.imshow(some_digit_img, cmap="binary")
plt.axis("off")
plt.show()

In [ ]:
y[0]

In [ ]:
import numpy as np
y = y.astype(np.uint8)
y

In [ ]:
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]
#the set provided is already shuffled and split into training/test sets
#this prevents missing digits from sets (due to it being shuffled)
#and prevents many instances of digits in a row (same)

In [ ]:
y_train_5 = (y_train == 5)
y_test_5 = (y_test == 5)

In [ ]:
#stochastic gradient descent (SGD)
from sklearn.linear_model import SGDClassifier
sgd_clf = SGDClassifier()#random_state=False for reproducible results, however it relies on randomness
sgd_clf.fit(X_train, y_train_5)

In [ ]:
print(y[11])
sgd_clf.predict([X[11]]) #can predict

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

skfolds = StratifiedKFold(n_splits=3)

for train_index, test_index in skfolds.split(X_train, y_train_5):
    clone_clf = clone(sgd_clf)
    X_train_folds = X_train[train_index]
    y_train_folds = y_train_5[train_index]
    X_test_folds = X_train[test_index]
    y_test_folds = y_train_5[test_index]
    
    clone_clf.fit(X_train_folds, y_train_folds)
    y_pred = clone_clf.predict(X_test_folds)

In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="accuracy")

In [ ]:
from sklearn.base import BaseEstimator

def Never5Classifier(BaseEstimator):
    def fit(self, X, y=None):
        return self
    def predict(self, X):
        return np.zeros((len(X), 1), dtype=bool)

In [ ]:
from sklearn.model_selection import cross_val_predict

y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)#returns the predictions made

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_train_5, y_train_pred)
#top row: non 5s (roughly 53k guesses correct)
#top right: incorrect guesses (in this case assumed to be 5)
#NOT TARGET[RIGHT, WRONG]
# IS TARGET[WRONG, RIGHT]
#bottom row: is 5s (roughly 4k guesses correct)

In [ ]:
y_train_perfect_predictions = y_train_5
confusion_matrix(y_train_5, y_train_perfect_predictions)
#this is an example of a perfect set of predictions

In [ ]:
#precision formula
#(TP) / (TP + FP)
#TP: true positives
#FP: false positives
5421 / (5421 + 1453) #precision for the confusion matrix above (predictions)

In [ ]:
#recall formula
#(TP) / (TP + FN)
5421 / (5421 + 1107) #for the predictions confusion matrix above

In [ ]:
#confusion matrix is as follows

#[TN, FP]
#[FN, TP]

In [ ]:
from sklearn.metrics import precision_score, recall_score

precision_score(y_train_5, y_train_pred)

In [ ]:
recall_score(y_train_5, y_train_pred)

In [ ]:
#f1 score
#combined precision and recall scores
#it is the harmonic mean of the precision and recall
from sklearn.metrics import f1_score

f1_score(y_train_5, y_train_pred)
#the f1 score favours when precision and recall are similar

#you cannot have both precision and recall
#increase in 1 will decrease the other
#depending on the situation, one may be greatly preferred than the other

In [ ]:
y_scores = sgd_clf.decision_function([some_digit])
y_scores

In [ ]:
threshold = 0
y_some_digit_pred = (y_scores > threshold)
y_some_digit_pred

In [ ]:
#to find which threshold to use
#cross validate

y_scores = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3, method="decision_function")
#the method tells it to return the decision scores instead of the predictions
y_scores

In [ ]:
y_scores

In [ ]:
#use precision recall curve to compute precision and recall for all possible threshold

from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)

def plot_precision_recall_vs_threshold(precisions, recalls, thresholds):
    plt.plot(thresholds, precisions[:-1], "b--", label="Precision")
    plt.plot(thresholds, recalls[:-1], "g-", label="Recall")
    
plot_precision_recall_vs_threshold(precisions, recalls, thresholds)
plt.show()

In [ ]:
threshold_90_precision = thresholds[np.argmax(precisions >= 0.90)]
threshold_90_precision #to find the threshold that gives 90% precision

In [ ]:
y_train_pred_90 = (y_scores >= threshold_90_precision)
y_train_pred_90
type(y_train_pred_90)

In [ ]:
precision_score(y_train_5, y_train_pred_90)

In [ ]:
recall_score(y_train_5, y_train_pred_90)

In [ ]:
#ROC - receiver operating characteristic
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_train_5, y_scores)

In [ ]:
def plot_roc_curve(fpr, tpr, label=None):
    plt.plot(fpr, tpr, linewidth=2, label=label, )
    plt.plot([0,1],[0,1],'k--')#dashed diagonal line
    
plot_roc_curve(fpr, tpr)
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.show()
#shows that the higher the recall (TPR)
#the more false positives (FPR)

#the dotted line representsROC curve of purely random classifier
#good classifier should aim as far away from it as possible (top left)

In [ ]:
#measure the Area Under Curve (AUC)

from sklearn.metrics import roc_auc_score

roc_auc_score(y_train_5, y_scores)
#perfect classifier will have a score of 1
#random classifier will have a score of 0.5

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_clf = RandomForestClassifier(random_state=42)
y_probas_forest = cross_val_predict(forest_clf, X_train, y_train_5, cv=3, method="predict_proba")

In [ ]:
y_scores_forest = y_probas_forest[:, 1] #score = proba of positive class
fpr_forest, tpr_forest, thresholds_forest = roc_curve(y_train_5, y_scores_forest)

In [ ]:
#plot the ROC curve

plt.plot(fpr, tpr, "b:", label="SGD")
plot_roc_curve(fpr_forest, tpr_forest, "Random forest")
plt.legend(loc="lower right")
plt.show()
#this shows that the random forest classifier is superior
#to the  SGD classifier because
#its ROC curve is closer to the top left (greater Area Under Curve AUC)

In [ ]:
roc_auc_score(y_train_5, y_scores_forest)

In [ ]:
#Support Vector Machine classifier
from sklearn.svm import SVC
svm_clf = SVC()
svm_clf.fit(X_train, y_train) #not y_train_5
svm_clf.predict([some_digit])
#the code trains the SVC using the original target classes (0-9) len:10
#behind the scenes it is doing OvO (one vs one)
#which means it trained 45 binary classifiers
#got their decision scores
#then chose the highest scoring class
#(that is why it takes so long)

In [ ]:
some_digit_scores = svm_clf_clf.decision_function([some_digit])
some_digit_scores

In [ ]:
np.argmax(some_digit_scores)

In [ ]:
svm_clf.classes_

In [ ]:
#np.argmax(some_digit_scores)
#returns class value which is the same index as
#the highest scoring decision_function
#in this case it is index 5, value 5
svm_clf.classes_[5]

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
ovr_clf = OneVsRestClassifier(SVC())
ovr_clf.fit(X_train, y_train)

In [ ]:
ovr_clf.predict([some_digit])

In [ ]:
len(ovr_clf.estimators_)

In [ ]:
ovr_clf.estimators_

In [ ]:
sgd_clf.fit(X_train, y_train)
sgd_clf.predict([some_digit])
#SGD classifiers can directly classify instances into multiple classes
#so OvO or OvR was not necessary

In [ ]:
sgd_clf.decision_function([some_digit])
#class 3 and 5 are very similar in this case

In [ ]:
cross_val_score(sgd_clf, X_train, y_train, cv=3, scoring="accuracy")
#the result is above 87% each time
#random classifier would get about 10%

In [ ]:
#to increase accuracy even more, it is worth scaling the inputs
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype(np.float64))
cross_val_score(sgd_clf, X_train_scaled, y_train, cv=3, scoring="accuracy")

In [ ]:
y_train_pred = cross_val_predict(sgd_clf, X_train_scaled, y_train, cv=3)
conf_mx = confusion_matrix(y_train, y_train_pred)
conf_mx

In [ ]:
plt.matshow(conf_mx, cmap=plt.cm.gray)
plt.show()
#the result is pretty good
#since most images are on the main diagonal
#5 is slightly darker than the rest
#this may be because there were fewer 5s in the dataset
#or because the classifier performs worse on the 5

In [ ]:
#divide each value in the confusion matrix
#by the number of images of the corresponding class
row_sums = conf_mx.sum(axis=1, keepdims=True)
norm_conf_mx = conf_mx / row_sums

In [ ]:
np.fill_diagonal(norm_conf_mx, 0)
plt.matshow(norm_conf_mx, cmap=plt.cm.gray)
plt.show()
#the column for 8 is bright
#which indicates that many numbers get incorrectly classified as 8
#while the row for 8 is dark, meaning 8 gets correctly classified

#the goal could be to reduce the amount of wrong 8 predictions
#possibly by increasing the number of images of numbers that look like 8
#(but are not)